# 03. EDA histórico de 52 preguntas

Este análisis conserva el corte validado 2012-2024 para que sus 52 hallazgos, el Word y el PDF sigan siendo reproducibles. La actualización 2012-2026 se utiliza en los notebooks 01, 02 y 04 para entrenamiento e inferencia.

In [1]:

from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Proyecto_Buenaventura_Final')
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'src' else Path.cwd().resolve()

os.environ['PROYECTO_BUENAVENTURA_ROOT'] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT / 'src')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Proyecto:', PROJECT_ROOT)


Proyecto: C:\Users\juanc\Desktop\Proyecto_Buenaventura_Final


## Visualizaciones principales

Las figuras completas también están disponibles en `reportes/figuras`.

![Serie mensual CIF](../reportes/figuras/01_serie_cif.png)

![Matriz de correlaciones](../reportes/figuras/03_correlaciones.png)

In [2]:
%run "eda_compute.py"

{
  "rows": 5625947,
  "months": 156,
  "backtest": {
    "Naive estacional (t-12)": {
      "MAE": 338898420.64,
      "RMSE": 386070205.62,
      "sMAPE_pct": 23.534,
      "WAPE_pct": 25.235
    },
    "Ridge": {
      "MAE": 97000318.2,
      "RMSE": 118478948.3,
      "sMAPE_pct": 7.251,
      "WAPE_pct": 7.223
    },
    "Random Forest": {
      "MAE": 204643600.41,
      "RMSE": 236648471.09,
      "sMAPE_pct": 14.654,
      "WAPE_pct": 15.238
    },
    "HistGradientBoosting": {
      "MAE": 296888913.74,
      "RMSE": 326959183.32,
      "sMAPE_pct": 20.394,
      "WAPE_pct": 22.107
    }
  }
}


## Pregunta 1. ¿Cuántas filas y columnas tiene el conjunto filtrado?

**Hallazgo:** El conjunto contiene 5.625.947 registros y 28 columnas después de filtrar la aduana 35.

**Implicación:** El volumen exige lectura por bloques en la extracción y un archivo analítico agregado por mes para el entrenamiento.

In [3]:
df.shape

(5625947, 28)

## Pregunta 2. ¿Qué tipos de datos quedaron después de la normalización?

**Hallazgo:** Las variables cuantitativas quedaron normalizadas como numéricas; fecha se convirtió a datetime y los identificadores se conservaron como códigos.

**Implicación:** Los códigos de país, capítulo y transporte deben tratarse como categorías, aunque estén representados con números.

In [4]:
df.dtypes.astype(str).value_counts()

Float64           14
Int64              9
object             3
int64              1
datetime64[ns]     1
Name: count, dtype: int64

## Pregunta 3. ¿Cuánta memoria ocupa el conjunto en memoria?

**Hallazgo:** La tabla ocupa aproximadamente 2.249,11 MB en pandas.

**Implicación:** En Colab conviene cargar el archivo filtrado, seleccionar columnas y construir agregados antes de modelar.

In [5]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(2249.1126251220703)

## Pregunta 4. ¿Cuál es la granularidad de una fila?

**Hallazgo:** Cada fila representa un registro o línea de importación asociado a un mes, una aduana, un país y una subpartida arancelaria; no representa un mes completo.

**Implicación:** El modelo de pronóstico debe trabajar con una tabla mensual agregada, no con líneas individuales mezcladas aleatoriamente.

In [6]:
df[['FECH','ADUA','PAISGEN','NABAN','VACID','PNK']].head()

,FECH,ADUA,PAISGEN,NABAN,VACID,PNK
0,1204,35,23,1108130000.0,18144.0,21000.0
1,1204,35,23,1901101000.0,63421.29,4896.0
2,1204,35,23,1901101000.0,100910.12,10233.6
3,1204,35,23,1901101000.0,118267.1,12096.0
4,1204,35,23,2512000000.0,5797.01,4875.0


## Pregunta 5. ¿Cuál es la cobertura temporal y existen meses faltantes?

**Hallazgo:** La serie cubre de 2012-01-01 a 2024-12-01, tiene 156 meses y no presenta meses ausentes.

**Implicación:** La continuidad permite usar rezagos de 1 a 12 meses sin reconstruir periodos faltantes.

In [7]:
monthly.index.min(), monthly.index.max(), len(monthly), monthly.isna().all(axis=1).sum()

(Timestamp('2012-01-01 00:00:00'),
 Timestamp('2024-12-01 00:00:00'),
 156,
 np.int64(0))

## Pregunta 6. ¿Existe una clave primaria y cuántos duplicados exactos hay?

**Hallazgo:** No existe un identificador único de línea. Se encontraron 4.543 coincidencias exactas (0,081 %).

**Implicación:** No deben eliminarse automáticamente: líneas comerciales distintas pueden compartir todos los campos publicados. Se auditarán por archivo y mes.

In [8]:
business_cols=[c for c in df if not c.startswith('SOURCE_') and c!='fecha']; df.duplicated(business_cols).sum()

np.int64(4543)

## Pregunta 7. ¿Qué columnas tienen valores nulos?

**Hallazgo:** Los nulos son bajos: SEGUROS 0,0040 %, FLETE 0,0018 %, VAFODO 0,0001 % y DEREL 0,0001 %.

**Implicación:** Se puede imputar cero solo cuando el diccionario confirme ausencia de cargo; para FOB se debe excluir o imputar con relación CIF-FOB.

In [9]:
(df.isna().mean().mul(100).sort_values(ascending=False).loc[lambda s:s>0])

SEGUROS    0.003964
FLETE      0.001777
VAFODO     0.000142
DEREL      0.000071
dtype: float64

## Pregunta 8. ¿Qué proporción de filas contiene al menos un nulo?

**Hallazgo:** Solo 0,005 % de las filas contiene al menos un valor nulo.

**Implicación:** La eliminación puntual tendría poco impacto, pero la decisión debe hacerse por variable y significado, no de forma global.

In [10]:
df.isna().any(axis=1).mean()*100

np.float64(0.004639218961714356)

## Pregunta 9. ¿El patrón de nulos puede clasificarse como MCAR, MAR o MNAR?

**Hallazgo:** No es posible probar MNAR con estos datos. La mayor variación anual de ausencia fue 0,031 puntos porcentuales en SEGUROS, por lo que el patrón es pequeño pero puede depender del periodo de reporte.

**Implicación:** Se tratará como potencialmente MAR por año y se incluirán indicadores de ausencia si la imputación mejora la validación temporal.

In [11]:
df.isna().groupby(df['fecha'].dt.year).mean().max().sub(df.isna().groupby(df['fecha'].dt.year).mean().min()).sort_values(ascending=False).head()

SEGUROS    0.000315
FLETE      0.000152
VAFODO     0.000010
DEREL      0.000007
PAISCOM    0.000000
dtype: float64

## Pregunta 10. ¿Existen columnas constantes?

**Hallazgo:** ADUA es constante porque el conjunto ya fue filtrado al código 35, Buenaventura.

**Implicación:** ADUA debe retirarse del conjunto de features: no aporta variación predictiva.

In [12]:
[c for c in df if df[c].nunique(dropna=False)<=1]

['ADUA']

## Pregunta 11. ¿Cuál es la cardinalidad de las variables principales?

**Hallazgo:** Hay 207 países de origen, 97 capítulos, 4 modos de transporte y una cardinalidad mucho mayor en subpartida.

**Implicación:** Se debe controlar la dimensionalidad con capítulos, top-N y agrupación 'Otros', en vez de one-hot de todas las subpartidas.

In [13]:
df[['PAISGEN','capitulo','VIATRANS','NABAN']].nunique()

PAISGEN      207
capitulo      97
VIATRANS       4
NABAN       7007
dtype: int64

## Pregunta 12. ¿Hubo errores de tipado al consolidar los años?

**Hallazgo:** Los archivos alternaban coma, punto y punto y coma, y algunos meses usaban punto de miles con coma decimal. Después de la normalización no quedaron conversiones numéricas inválidas.

**Implicación:** La función de ingestión debe conservar la detección de separador y la conversión localizada para futuras actualizaciones.

In [14]:
df[['FECH','VACID','PNK','VAFODO']].dtypes

FECH        Int64
VACID     Float64
PNK       Float64
VAFODO    Float64
dtype: object

## Pregunta 13. ¿Hay categorías mal escritas o inconsistentes?

**Hallazgo:** Las categorías analíticas son códigos, por lo que no se detectan faltas ortográficas en ellas. Sí hubo nombres de archivos y rutas mensuales inconsistentes.

**Implicación:** Las etiquetas legibles deben incorporarse desde catálogos oficiales mediante joins; no deben inferirse desde nombres de archivos.

In [15]:
df[['PAISGEN','capitulo','VIATRANS']].astype('Int64').nunique()

PAISGEN     207
capitulo     97
VIATRANS      4
dtype: int64

## Pregunta 14. ¿Existen valores imposibles?

**Hallazgo:** No hay pesos ni valores CIF negativos. Se encontró un VADUA negativo y una BASEIVA negativa, casos aislados que requieren revisión documental.

**Implicación:** Esos dos registros deben marcarse y excluirse de features monetarias derivadas si no corresponden a ajustes válidos.

In [16]:
{c:int((df[c]<0).sum()) for c in ['PNK','VACID','VADUA','BASEIVA','DEREL']}

{'PNK': 0, 'VACID': 0, 'VADUA': 1, 'BASEIVA': 1, 'DEREL': 0}

## Pregunta 15. ¿Cuál será la variable objetivo y el horizonte de predicción?

**Hallazgo:** El objetivo principal es el valor CIF mensual en dólares, calculado como suma de VACID. El horizonte propuesto es un mes adelante.

**Implicación:** Cada fila de entrenamiento representará un mes y solo usará información disponible hasta el cierre del mes anterior.

In [17]:
monthly['cif_usd']=df.groupby('fecha')['VACID'].sum(); target=monthly['cif_usd']

## Pregunta 16. ¿Cómo se distribuye el target mensual?

**Hallazgo:** La media es US$ 1.118.309.840,00, la mediana US$ 1.037.362.529,41, el mínimo US$ 693.549.752,38 y el máximo US$ 1.938.313.091,96.

**Implicación:** Se debe evaluar con métricas absolutas y porcentuales para que los meses de mayor valor no dominen toda la evaluación.

In [18]:
monthly['cif_usd'].describe()

count                156.0
mean     1118309840.004808
std       260473044.370423
min           693549752.38
25%           935244129.78
50%          1037362529.41
75%          1217618464.02
max          1938313091.96
Name: cif_usd, dtype: Float64

## Pregunta 17. ¿El target presenta asimetría o curtosis?

**Hallazgo:** La asimetría es 1,2477 y la curtosis excedente 1,0351.

**Implicación:** La distribución es sesgada a la derecha; conviene comparar entrenamiento en escala original y logarítmica.

In [19]:
monthly['cif_usd'].agg(['skew','kurt'])

skew    1.247693
kurt    1.035132
Name: cif_usd, dtype: float64

## Pregunta 18. ¿Qué meses son outliers del target?

**Hallazgo:** El criterio IQR identifica ocho meses, concentrados entre diciembre de 2021 y septiembre de 2022; z-score mayor que 3 identifica marzo y agosto de 2022.

**Implicación:** No se eliminarán: son periodos reales de expansión. Se usarán modelos robustos, indicadores de régimen y análisis de error por periodo.

In [20]:
s=monthly['cif_usd']; q1,q3=s.quantile([.25,.75]); iqr=q3-q1; s[(s<q1-1.5*iqr)|(s>q3+1.5*iqr)]

fecha
2021-12-01    1857235453.09
2022-01-01    1683739069.57
2022-02-01     1667045237.4
2022-03-01    1925686740.12
2022-06-01    1737501386.79
2022-07-01    1734319133.23
2022-08-01    1938313091.96
2022-09-01    1889318447.29
Name: cif_usd, dtype: Float64

## Pregunta 19. ¿Hay meses con target nulo?

**Hallazgo:** El target tiene 0 meses nulos.

**Implicación:** No se requiere imputación temporal del objetivo; cualquier mes nuevo incompleto deberá bloquearse antes de inferencia.

In [21]:
monthly['cif_usd'].isna().sum()

np.int64(0)

## Pregunta 20. ¿Qué variables provocarían fuga de información?

**Hallazgo:** FOB contemporáneo correlaciona 0,9971 con CIF y flete contemporáneo 0,9066. Ambos forman parte del mismo mes que se intenta predecir.

**Implicación:** Solo se usarán sus rezagos. Utilizar valores contemporáneos produciría data leakage y una precisión irreal.

In [22]:
monthly.corr(method='spearman')['cif_usd'].sort_values(ascending=False)

cif_usd                    1.000000
fob_usd                    0.997082
flete_usd                  0.906646
seguros_usd                0.773112
precio_implicito_usd_kg    0.683305
registros                  0.619899
anio                       0.535546
peso_neto_kg               0.518098
paises_origen              0.283444
mes                        0.109027
capitulos                 -0.095257
Name: cif_usd, dtype: float64

## Pregunta 21. ¿Una transformación logarítmica mejora la forma del target?

**Hallazgo:** La asimetría baja de 1,2477 a 0,7991 con log1p.

**Implicación:** Ridge y modelos de boosting deben probarse sobre log1p(CIF), revirtiendo con expm1 y recortando predicciones negativas.

In [23]:
monthly['cif_usd'].skew(), np.log1p(monthly['cif_usd']).skew()

(np.float64(1.2476931198110042), np.float64(0.7990906488236116))

## Pregunta 22. ¿El target conserva memoria temporal?

**Hallazgo:** La autocorrelación es 0,8712 en t-1 y 0,3849 en t-12.

**Implicación:** Los rezagos recientes son esenciales y el rezago anual debe conservarse como referencia estacional.

In [24]:
from statsmodels.tsa.stattools import acf; acf(monthly['cif_usd'],nlags=12)[[1,12]]

array([0.87120244, 0.38489083])

## Pregunta 23. ¿Cuáles son los estadísticos descriptivos de las variables monetarias y de peso?

**Hallazgo:** Por registro, la mediana de PNK es 757,67 kg y la de VACID es US$ 4.977,00; las medias son mucho mayores por la cola derecha.

**Implicación:** Para el nivel de registro se prefieren medianas, cuantiles y transformaciones robustas, no solo medias.

In [25]:
df[['PNK','VAFODO','FLETE','VACID','SEGUROS']].describe().T

,count,mean,std,min,25%,50%,75%,max
PNK,5625947.0,25607.816048,314947.051181,0.0,82.98,757.67,7500.0,58500000.0
VAFODO,5625939.0,28775.676132,125486.214672,0.0,533.45,4563.0,23979.39,63974565.18
FLETE,5625847.0,2122.527807,20708.266036,0.0,21.94,181.67,1323.285,39300899.0
VACID,5625947.0,31009.238985,138552.038002,0.01,588.475,4977.0,26126.155,92231312.0
SEGUROS,5625724.0,42.694892,4214.091046,0.0,0.6,4.58,25.94,9897015.0


## Pregunta 24. ¿Cómo son las distribuciones de las variables numéricas?

**Hallazgo:** Las variables de peso, valor, flete y seguros presentan colas largas y diferencias marcadas entre mediana y extremos.

**Implicación:** El modelado mensual debe usar sumas y razones estables; si se modela por segmento, se aplicará log1p y winsorización solo dentro del train.

In [26]:
df[['PNK','VAFODO','FLETE','VACID','SEGUROS']].quantile([.01,.25,.5,.75,.99])

,PNK,VAFODO,FLETE,VACID,SEGUROS
0.01,0.26,3.2,0.09,3.92,0.01
0.25,82.98,533.45,21.94,588.475,0.6
0.50,757.67,4563.0,181.67,4977.0,4.58
0.75,7500.0,23979.39,1323.285,26126.155,25.94
0.99,289934.08,354693.586,28759.27,380691.3758,495.88


## Pregunta 25. ¿Qué asimetría y curtosis presentan las variables numéricas?

**Hallazgo:** En registros, VACID tiene asimetría 98,8494; FLETE 1.222,8707 y SEGUROS 2.304,9185.

**Implicación:** Los modelos lineales no deben recibir estos valores crudos sin transformación o agregación.

In [27]:
pd.DataFrame({'skew':df[['PNK','VAFODO','FLETE','VACID','SEGUROS']].skew(),'kurt':df[['PNK','VAFODO','FLETE','VACID','SEGUROS']].kurt()})

,skew,kurt
PNK,55.861263,5881.391731
VAFODO,67.49974,21392.180847
FLETE,1222.870735,2305800.578173
VACID,98.849414,46468.326759
SEGUROS,2304.918524,5408167.999793


## Pregunta 26. ¿Cuántos outliers detecta IQR en variables de registro?

**Hallazgo:** IQR marca 875.016 pesos netos, 571.693 valores CIF y 664.484 fletes.

**Implicación:** IQR no debe usarse como regla de borrado en comercio exterior; servirá para diagnóstico y transformaciones robustas.

In [28]:
{c:((df[c]<df[c].quantile(.25)-1.5*(df[c].quantile(.75)-df[c].quantile(.25)))|(df[c]>df[c].quantile(.75)+1.5*(df[c].quantile(.75)-df[c].quantile(.25)))).sum() for c in ['PNK','VACID','FLETE']}

{'PNK': np.int64(875016), 'VACID': np.int64(571693), 'FLETE': np.int64(664484)}

## Pregunta 27. ¿Qué outliers mensuales detecta z-score?

**Hallazgo:** Solo marzo y agosto de 2022 superan tres desviaciones estándar en la serie mensual.

**Implicación:** Se conservarán y se medirá el error del modelo en esos meses para evaluar robustez ante picos.

In [29]:
s=monthly['cif_usd']; monthly.index[np.abs(stats.zscore(s))>3]

DatetimeIndex(['2022-03-01', '2022-08-01'], dtype='datetime64[ns]', name='fecha', freq=None)

## Pregunta 28. ¿Existen escalas numéricas muy dispares?

**Hallazgo:** La razón entre la mayor y menor media de las variables analizadas es aproximadamente 2.360.159,46.

**Implicación:** Ridge requiere StandardScaler ajustado solo con train; los modelos de árboles no requieren escalado.

In [30]:
df[['SEGUROS','FLETE','PNK','VACID','VACIP']].mean().sort_values()

SEGUROS           42.694892
FLETE           2122.527807
PNK            25607.816048
VACID          31009.238985
VACIP      100153369.225007
dtype: Float64

## Pregunta 29. ¿Hay ceros excesivos?

**Hallazgo:** DEREL contiene 46,232 % de ceros y TOTALIVAYO 6,072 %; en las demás variables principales la proporción es inferior a 0,2 %.

**Implicación:** Se crearán indicadores binarios de arancel cero y tributo cero cuando se modele composición; no se imputarán esos ceros.

In [31]:
df[['PNK','VAFODO','FLETE','VACID','TOTALIVAYO','SEGUROS','DEREL']].eq(0).mean().mul(100)

PNK            0.000018
VAFODO         0.034785
FLETE          0.152262
VACID               0.0
TOTALIVAYO     6.071956
SEGUROS        0.144852
DEREL         46.231947
dtype: Float64

## Pregunta 30. ¿Qué transformaciones numéricas son razonables?

**Hallazgo:** log1p reduce las colas y acepta ceros. Box-Cox no es directamente aplicable a variables con cero y aportaría menos interpretabilidad.

**Implicación:** Se usará log1p para target y montos rezagados; porcentajes y razones se limitarán con cuantiles aprendidos en train.

In [32]:
np.log1p(df[['PNK','VAFODO','FLETE','VACID','SEGUROS']]).skew()

PNK        -0.13804
VAFODO    -0.504331
FLETE      -0.09904
VACID     -0.499076
SEGUROS    0.624895
dtype: Float64

## Pregunta 31. ¿Qué variables mensuales están asociadas con CIF?

**Hallazgo:** FOB, flete y seguros mensuales muestran correlaciones de 0,9971, 0,9066 y 0,7731 con CIF; peso neto alcanza 0,5181.

**Implicación:** Estas relaciones sirven para features rezagadas; las versiones del mismo mes se excluyen por fuga.

In [33]:
monthly.select_dtypes('number').corr(method='spearman')['cif_usd'].sort_values(ascending=False)

cif_usd                    1.000000
fob_usd                    0.997082
flete_usd                  0.906646
seguros_usd                0.773112
precio_implicito_usd_kg    0.683305
registros                  0.619899
anio                       0.535546
peso_neto_kg               0.518098
paises_origen              0.283444
mes                        0.109027
capitulos                 -0.095257
Name: cif_usd, dtype: float64

## Pregunta 32. ¿La serie es estacionaria?

**Hallazgo:** ADF en nivel da p=0.481040; en primera diferencia logarítmica da p<0.000001.

**Implicación:** La serie en nivel no es estacionaria. ARIMA/SARIMA requeriría diferenciación; modelos supervisados usarán tendencia y rezagos.

In [34]:
from statsmodels.tsa.stattools import adfuller; adfuller(monthly['cif_usd'])[1], adfuller(np.log1p(monthly['cif_usd']).diff().dropna())[1]

(np.float64(0.48104010135139114), np.float64(2.662203718336997e-10))

## Pregunta 33. ¿Qué países concentran el mayor valor CIF?

**Hallazgo:** El código 215 lidera con US$ 66,18 mil millones; le siguen 493 y 249 con US$ 18,89 y US$ 18,75 mil millones.

**Implicación:** Se deben crear participaciones rezagadas de los países principales y agrupar el resto.

In [35]:
df.groupby('PAISGEN')['VACID'].sum().nlargest(10)

PAISGEN
215    66182428873.709999
493    18894373789.560001
249        18754315935.43
190         7923902178.09
211         6957286140.16
589         6909847547.88
399         5954004848.22
361          5652618446.2
149         4576704782.38
97          4213845701.12
Name: VACID, dtype: Float64

## Pregunta 34. ¿Qué peso tienen las categorías raras de país?

**Hallazgo:** Los países con menos de 100 registros representan apenas 0,026 % de las filas.

**Implicación:** Las categorías raras pueden agruparse en 'Otros' sin perder volumen significativo.

In [36]:
vc=df['PAISGEN'].value_counts(); vc[vc<100].sum()/len(df)*100

np.float64(0.025737889105603023)

## Pregunta 35. ¿Existe alta cardinalidad categórica?

**Hallazgo:** País tiene 207 categorías y capítulo 97; subpartida es mucho más granular.

**Implicación:** Se evitará one-hot de subpartida. Para modelos globales se usarán capítulo, top-N y estadísticas históricas rezagadas.

In [37]:
df[['PAISGEN','capitulo','NABAN']].nunique()

PAISGEN      207
capitulo      97
NABAN       7007
dtype: int64

## Pregunta 36. ¿Qué capítulos arancelarios concentran el valor?

**Hallazgo:** Los capítulos 85, 84, 87, 39 y 10 son los cinco principales; juntos explican 39,692 % del CIF.

**Implicación:** Se crearán participaciones mensuales rezagadas para estos capítulos y una categoría residual.

In [38]:
df.groupby('capitulo')['VACID'].sum().nlargest(10)

capitulo
85    16538113606.049999
84    15695436376.370001
87    13832600540.790001
39    11945386920.700001
10        11233176061.51
72         8659768783.91
29         6142487712.76
31          5133751989.4
23          4356805395.3
40         4262216824.75
Name: VACID, dtype: Float64

## Pregunta 37. ¿Qué relación tienen las categorías principales con el target?

**Hallazgo:** Los cinco países principales concentran 68,047 % del CIF, mientras los cinco capítulos principales concentran 39,692 %.

**Implicación:** La composición por país tiene mayor poder potencial, pero las participaciones deben calcularse con meses anteriores.

In [39]:
shares=df.groupby('PAISGEN')['VACID'].sum().sort_values(ascending=False); shares.head(5).sum()/shares.sum()*100

np.float64(68.04700264351013)

## Pregunta 38. ¿Qué estrategia de encoding conviene?

**Hallazgo:** Un top-15 más 'Otros' reduce país a 16 categorías; para capítulo puede emplearse top-20 más 'Otros'.

**Implicación:** El one-hot se ajustará dentro del pipeline. Target encoding solo sería válido con codificación out-of-fold y respetando el tiempo.

In [40]:
top=df['PAISGEN'].value_counts().head(15).index; pd.Series(np.where(df['PAISGEN'].isin(top),df['PAISGEN'].astype(str),'OTROS')).nunique()

16

## Pregunta 39. ¿Cuál es la matriz de correlación mensual?

**Hallazgo:** CIF se mueve casi en paralelo con FOB y presenta asociación moderada con registros y peso.

**Implicación:** La matriz confirma qué variables deben rezagarse y ayuda a evitar duplicar señales contemporáneas.

In [41]:
monthly[['cif_usd','fob_usd','peso_neto_kg','flete_usd','seguros_usd','registros']].corr(method='spearman')

,cif_usd,fob_usd,peso_neto_kg,flete_usd,seguros_usd,registros
cif_usd,1.000000,0.997082,0.518098,0.906646,0.773112,0.619899
fob_usd,0.997082,1.000000,0.536268,0.880261,0.778941,0.635783
peso_neto_kg,0.518098,0.536268,1.000000,0.386929,0.311046,0.462936
flete_usd,0.906646,0.880261,0.386929,1.000000,0.722913,0.447744
seguros_usd,0.773112,0.778941,0.311046,0.722913,1.000000,0.346965
registros,0.619899,0.635783,0.462936,0.447744,0.346965,1.000000


## Pregunta 40. ¿Qué rezagos se correlacionan más con el target?

**Hallazgo:** La media móvil CIF de 3 meses correlaciona 0,8501; CIF t-1, 0,8280; CIF t-2, 0,7838 y CIF t-3, 0,7735.

**Implicación:** El núcleo del modelo debe priorizar rezagos recientes y medias móviles calculadas con shift(1).

In [42]:
model_df.corr(method='spearman')['target'].sort_values(ascending=False).head(10)

target          1.000000
cif_media_3     0.850076
cif_lag_1       0.827956
cif_media_6     0.798621
cif_lag_2       0.783775
cif_lag_3       0.773463
cif_media_12    0.731304
cif_lag_6       0.627679
tendencia       0.576240
cif_lag_12      0.499811
Name: target, dtype: float64

## Pregunta 41. ¿Existe multicolinealidad entre rezagos?

**Hallazgo:** Todos los VIF evaluados son altos; las medias móviles de 3 y 6 meses superan 2.000 por combinar los mismos rezagos.

**Implicación:** Ridge puede regularizar; en modelos lineales explicativos se seleccionará una sola ventana o se aplicará PCA.

In [43]:
pd.Series(vif).sort_values(ascending=False)

cif_media_6     2085.318279
cif_media_3     2057.701663
cif_media_12     973.492849
cif_lag_3        394.264598
cif_lag_1        349.358372
cif_lag_6        231.280271
cif_lag_12       108.364821
dtype: float64

## Pregunta 42. ¿Qué interacciones pueden aportar señal?

**Hallazgo:** El costo logístico relativo y el precio implícito conectan valor, peso y flete de forma interpretable.

**Implicación:** Se crearán flete/FOB, CIF/kg y sus cambios, siempre rezagados para el horizonte t+1.

In [44]:
monthly.assign(flete_pct=monthly.flete_usd/monthly.fob_usd,precio=monthly.cif_usd/monthly.peso_neto_kg)[['flete_pct','precio']].describe()

,flete_pct,precio
count,156.0,156.0
mean,0.070007,1.214765
std,0.025058,0.222849
min,0.043247,0.697476
25%,0.055924,1.063266
50%,0.060946,1.191518
75%,0.069351,1.350227
max,0.169905,1.799778


## Pregunta 43. ¿Qué variables son redundantes?

**Hallazgo:** FOB contemporáneo es casi redundante con CIF (0,9971); las distintas medias móviles también son redundantes entre sí.

**Implicación:** Se limitará la duplicación de ventanas y se comparará el rendimiento por ablación.

In [45]:
monthly[['cif_usd','fob_usd','flete_usd','seguros_usd']].corr(method='spearman')

,cif_usd,fob_usd,flete_usd,seguros_usd
cif_usd,1.000000,0.997082,0.906646,0.773112
fob_usd,0.997082,1.000000,0.880261,0.778941
flete_usd,0.906646,0.880261,1.000000,0.722913
seguros_usd,0.773112,0.778941,0.722913,1.000000


## Pregunta 44. ¿Qué variables muestran poco poder predictivo individual?

**Hallazgo:** mes_cos tiene correlación -0,0056; peso t-12, 0,1137; y mes_sin, -0,1259 en términos absolutos.

**Implicación:** No se eliminarán solo por correlación: pueden aportar de forma no lineal, pero deberán justificar su permanencia en validación.

In [46]:
model_df.corr(method='spearman')['target'].abs().sort_values().head()

mes_cos        0.005599
peso_lag_12    0.113721
mes_sin        0.125902
peso_lag_6     0.187413
peso_lag_3     0.239627
Name: target, dtype: float64

## Pregunta 45. ¿Existe una tendencia de largo plazo?

**Hallazgo:** La pendiente lineal es US$ 3.370.725,70 por mes, con R²=0,3418 y p<0,001.

**Implicación:** Se incluirá índice temporal o tendencia, y se vigilará que no extrapole sin límites.

In [47]:
stats.linregress(np.arange(len(monthly)),monthly['cif_usd'])

LinregressResult(slope=np.float64(3370725.703506678), intercept=np.float64(857078597.9830402), rvalue=np.float64(0.5846313890946316), pvalue=np.float64(1.1200926176611488e-15), stderr=np.float64(376931.026766268), intercept_stderr=np.float64(33785652.11707976))

## Pregunta 46. ¿Existe estacionalidad mensual?

**Hallazgo:** Agosto tiene el índice medio más alto, 1,0709; junio el más bajo, 0,9469. La amplitud media es moderada.

**Implicación:** Se usarán seno/coseno del mes y rezago 12; la estacionalidad se validará por ventanas móviles.

In [48]:
monthly.groupby(monthly.index.month)['cif_usd'].mean()/monthly['cif_usd'].mean()

fecha
1     0.963353
2     0.948417
3     0.971516
4     0.985359
5     0.998715
6     0.946869
7     1.000912
8     1.070903
9     1.021001
10    1.058004
11     1.01677
12    1.018182
Name: cif_usd, dtype: Float64

## Pregunta 47. ¿Hay drift entre el periodo histórico y el reciente?

**Hallazgo:** KS=0,8528 con p<0,001; la mediana de 2022-2024 es 45,836 % mayor que la de 2012-2021.

**Implicación:** La validación debe dar mayor peso a ventanas recientes y el modelo debe reentrenarse al incorporar nuevos meses.

In [49]:
stats.ks_2samp(monthly.loc[:'2021-12','cif_usd'],monthly.loc['2022-01':,'cif_usd'])

KstestResult(statistic=np.float64(0.8527777777777777), pvalue=np.float64(1.091893425534393e-21), statistic_location=np.float64(1174895412.36), statistic_sign=np.int8(1))

## Pregunta 48. ¿Qué segmentos deben vigilarse por separado?

**Hallazgo:** Los países 215, 493 y 249 dominan el origen; los capítulos 85, 84 y 87 lideran productos.

**Implicación:** El producto debe mostrar pronóstico total y contribuciones de estos segmentos, con alerta cuando cambie su participación.

In [50]:
df.groupby('PAISGEN')['VACID'].sum().nlargest(5), df.groupby('capitulo')['VACID'].sum().nlargest(5)

(PAISGEN
 215    66182428873.709999
 493    18894373789.560001
 249        18754315935.43
 190         7923902178.09
 211         6957286140.16
 Name: VACID, dtype: Float64,
 capitulo
 85    16538113606.049999
 84    15695436376.370001
 87    13832600540.790001
 39    11945386920.700001
 10        11233176061.51
 Name: VACID, dtype: Float64)

## Pregunta 49. ¿Qué features finales se proponen?

**Hallazgo:** Se proponen rezagos CIF 1, 2, 3, 6 y 12; rezagos de peso, flete y registros; medias 3 y 12; calendario, tendencia y participaciones top-N rezagadas.

**Implicación:** El conjunto final se decidirá por validación walk-forward y ablación, no por correlación aislada.

In [51]:
feature_set=['cif_lag_1','cif_lag_2','cif_lag_3','cif_lag_6','cif_lag_12','peso_lag_1','peso_lag_3','flete_lag_1','registros_lag_1','media_3','media_12','mes_sin','mes_cos','tendencia']; feature_set

['cif_lag_1',
 'cif_lag_2',
 'cif_lag_3',
 'cif_lag_6',
 'cif_lag_12',
 'peso_lag_1',
 'peso_lag_3',
 'flete_lag_1',
 'registros_lag_1',
 'media_3',
 'media_12',
 'mes_sin',
 'mes_cos',
 'tendencia']

## Pregunta 50. ¿Cuál debe ser el pipeline de preprocesamiento?

**Hallazgo:** El pipeline debe validar integridad, normalizar formatos, agregar por mes, crear rezagos, imputar y escalar dentro de cada fold.

**Implicación:** Toda transformación que aprenda parámetros debe ajustarse solo con entrenamiento para impedir fuga.

In [52]:
pipeline=['validar_mes_completo','normalizar_decimal','agregar_mes','crear_rezagos_con_shift','imputar_train','escalar_train','modelo']; pipeline

['validar_mes_completo',
 'normalizar_decimal',
 'agregar_mes',
 'crear_rezagos_con_shift',
 'imputar_train',
 'escalar_train',
 'modelo']

## Pregunta 51. ¿Cómo debe hacerse el split y la validación?

**Hallazgo:** El backtest final usa 132 meses hasta diciembre de 2022 para train y 24 meses de 2023-2024 para prueba; dentro de train se aplicarán cortes expanding-window.

**Implicación:** No se usará train_test_split aleatorio. Cada predicción debe construirse únicamente con meses anteriores.

In [53]:
train=monthly.loc[:'2022-12']; test=monthly.loc['2023-01':'2024-12']; len(train),len(test)

(132, 24)

## Pregunta 52. ¿Qué modelos y métricas resultan adecuados?

**Hallazgo:** En el holdout 2023-2024, Ridge logró MAE US$ 97,00 millones, RMSE US$ 118,48 millones, sMAPE 7,251 % y WAPE 7,223 %. El naive estacional tuvo WAPE 25,235 %.

**Implicación:** Ridge es el candidato inicial por precisión e interpretabilidad; se comparará con SARIMA y boosting. Las métricas principales serán WAPE y MAE, con RMSE como penalización de errores grandes.

In [54]:
pd.DataFrame(backtest).T.sort_values('WAPE_pct')

,MAE,RMSE,sMAPE_pct,WAPE_pct
Ridge,9.700032e+07,1.184789e+08,7.251,7.223
Random Forest,2.046436e+08,2.366485e+08,14.654,15.238
HistGradientBoosting,2.968889e+08,3.269592e+08,20.394,22.107
Naive estacional (t-12),3.388984e+08,3.860702e+08,23.534,25.235
